In [2]:
# Install all required packages (corrected supabase name)
!pip install supabase xarray netCDF4 pandas pyarrow sentence-transformers

# Optional: upgrade pip if you see any weird errors later
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [5]:
from google.colab import userdata
from supabase import create_client, Client

supabase_url = userdata.get('SUPABASE_URL')     # ← Use the secret name you defined
supabase_key = userdata.get('SUPABASE_ANON_KEY') # ← Use the secret name you defined

if not supabase_url or not supabase_key:
    raise ValueError("Secrets not found! Check Colab Secrets panel.")

supabase: Client = create_client(supabase_url, supabase_key)

# Quick test
print("Connected to Supabase:", supabase_url)
buckets = supabase.storage.list_buckets()
print("Buckets:", [b.name for b in buckets])

Connected to Supabase: https://zutwiowxndmfcusxwyuc.supabase.co
Storage endpoint URL should have a trailing slash.
Buckets: []


In [12]:
buckets = supabase.storage.list_buckets()
print("Buckets now:", [b.name for b in buckets])

Buckets now: []


In [13]:
from supabase import create_client
from google.colab import userdata

supabase_url = userdata.get('SUPABASE_URL')
supabase_key = userdata.get('SUPABASE_ANON_KEY')
supabase = create_client(supabase_url, supabase_key)

# Optional: silence trailing slash warning
supabase.storage._client.base_url = supabase_url.rstrip('/') + '/storage/v1/'

# Create a tiny test file
test_content = b"Hello from Colab! This is a test upload."
test_path = "test/test_file.txt"

try:
    response = supabase.storage.from_("argo-parquet").upload(
        path=test_path,
        file=test_content,
        file_options={"content-type": "text/plain"}
    )
    print("Upload SUCCESS! Response:", response)
except Exception as e:
    print("Upload FAILED:", str(e))

# If upload works, try to get public URL
if "Upload SUCCESS" in locals():
    public_url = supabase.storage.from_("argo-parquet").get_public_url(test_path)
    print("Public URL (open in browser):", public_url)

Storage endpoint URL should have a trailing slash.
Upload SUCCESS! Response: UploadResponse(path='test/test_file.txt', full_path='argo-parquet/test/test_file.txt', fullPath='argo-parquet/test/test_file.txt')


In [28]:
import os
import xarray as xr
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from supabase import create_client, Client
from sentence_transformers import SentenceTransformer
from datetime import datetime, timedelta
import json
from google.colab import userdata
import numpy as np

# Initialize Supabase (use your secrets)
supabase_url = userdata.get('SUPABASE_URL')
supabase_key = userdata.get('SUPABASE_ANON_KEY')
supabase: Client = create_client(supabase_url, supabase_key)

# Load embedder (global, run once)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Initialization complete!")

Initialization complete!


In [29]:
def download_and_open_netcdf(url):
    filename = os.path.basename(url)
    !wget -O {filename} {url}

    ds = xr.open_dataset(filename, decode_cf=True, decode_times=False)  # Manual time decode
    print(f"Profiles: {ds.sizes['N_PROF']}, Format: {ds.attrs.get('FORMAT_VERSION', '3.1')}")

    return ds, filename

# Example call (replace URL if needed)
sample_url = "https://data-argo.ifremer.fr/geo/indian_ocean/2025/12/20251231_prof.nc"
ds, filename = download_and_open_netcdf(sample_url)

--2026-02-01 11:20:55--  https://data-argo.ifremer.fr/geo/indian_ocean/2025/12/20251231_prof.nc
Resolving data-argo.ifremer.fr (data-argo.ifremer.fr)... 134.246.232.86
Connecting to data-argo.ifremer.fr (data-argo.ifremer.fr)|134.246.232.86|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6174536 (5.9M) [application/x-netcdf]
Saving to: ‘20251231_prof.nc’

20251231_prof.nc    100%[===================>]   5.89M  5.15MB/s    in 1.1s    

2026-02-01 11:20:57 (5.15 MB/s) - ‘20251231_prof.nc’ saved [6174536/6174536]

Profiles: 89, Format: 3.1


In [37]:
profile = ds.isel(N_PROF=0)

print("DATA_MODE:", profile.DATA_MODE.values)
print("PI_NAME:", profile.PI_NAME.values)
print("PLATFORM_TYPE:", profile.PLATFORM_TYPE.values)


DATA_MODE: b'R'
PI_NAME: b'M Ravichandran                                                  '
PLATFORM_TYPE: b'ARVOR                           '


In [33]:
from datetime import datetime, UTC # Import UTC directly
import pandas as pd

def process_file_info(ds, filename):
    utc_now = datetime.now(UTC).isoformat() # Use UTC directly
    file_metadata = {
        "file_name": filename,
        "data_type": "Core-Argo profile",
        "format_version": str(ds.attrs.get("FORMAT_VERSION", "3.1")),
        "handbook_version": "Argo User's manual, June 2025",
        "reference_date_time": pd.to_datetime(ds.attrs.get("REFERENCE_DATE_TIME", "19500101000000"), format="%Y%m%d%H%M%S", errors='coerce').isoformat(),
        "data_mode": str(ds.attrs.get("DATA_MODE", "")),
        "data_state_indicator": str(ds.attrs.get("DATA_STATE_INDICATOR", "")),
        "project_name": str(ds.attrs.get("PROJECT_NAME", "Argo")),
        "dc_reference": str(ds.attrs.get("DC_REFERENCE", "")),
        "date_creation": pd.to_datetime(ds.attrs.get("DATE_CREATION", utc_now), errors='coerce').isoformat(),
        "date_update": pd.to_datetime(ds.attrs.get("DATE_UPDATE", utc_now), errors='coerce').isoformat(),
        "data_centre": str(ds.attrs.get("DATA_CENTRE", "")),
        "notes": str(ds.attrs.get("NOTES", "")),
        "float_count": len(set(ds.PLATFORM_NUMBER.values.ravel().astype(str))),
        "profile_count": ds.sizes["N_PROF"]
    }

    result = supabase.table("file_info").insert(file_metadata).execute()
    file_id = result.data[0]["file_id"]
    print(f"File inserted: {file_id}")
    return file_id

# Call it
file_id = process_file_info(ds, filename)

File inserted: da7c5e2c-d193-4520-a7db-965cae4e5328


In [36]:
import numpy as np
import pandas as pd

def get_safe_string_from_xarray_var(x):
    """Safely extract a clean string from xarray DataArray or scalar.Handles byte strings, char arrays, and scalars."""
    if x is None:
        return None
    # If it's an xarray DataArray
    if hasattr(x, "values"):
        val = x.values
        # Scalar (0-d array) → most common case
        if val.ndim == 0:
            val = val.item()
        # Char array (e.g. ['A','P','E','X'])
        elif val.dtype.kind in ("S", "U"):
            return "".join(val.astype(str)).strip()
        # Anything else → take first non-null
        else:
            try:
                val = val.flatten()[0]
            except Exception:
                return None
        # Decode bytes
        if isinstance(val, bytes):
            return val.decode("utf-8", errors="ignore").strip()
        return str(val).strip()
    # Plain python scalar
    if isinstance(x, bytes):
        return x.decode("utf-8", errors="ignore").strip()
    return str(x).strip()

def process_profile_metadata(ds, file_id):
    for i in range(ds.sizes["N_PROF"]):
        profile = ds.isel(N_PROF=i)
        float_id = get_safe_string_from_xarray_var(profile.PLATFORM_NUMBER.values.item())
        cycle_number = int(profile.CYCLE_NUMBER.values.item())

        # JULD
        ref_str = ds.attrs.get("REFERENCE_DATE_TIME", "19500101000000")
        try:
            juld_ref = pd.to_datetime(ref_str, format="%Y%m%d%H%M%S")
        except:
            juld_ref = pd.to_datetime("1950-01-01")

        raw_juld = profile.JULD.values.item()
        if pd.isna(raw_juld) or raw_juld >= 999999.0 or raw_juld < 0:
            juld = None
        else:
            juld = juld_ref + pd.Timedelta(days=float(raw_juld))

        metadata = {
            "float_id": float_id,
            "cycle_number": cycle_number,
            "file_id": file_id,
            "platform_number": float_id,
            "platform_type": get_safe_string_from_xarray_var(profile.PLATFORM_TYPE) if 'PLATFORM_TYPE' in profile else None,
            "pi_name": get_safe_string_from_xarray_var(profile.PI_NAME) if 'PI_NAME' in profile else None,
            "juld": juld.isoformat() if juld else None,
            "juld_qc": get_safe_string_from_xarray_var(profile.JULD_QC) if 'JULD_QC' in profile else None,
            "latitude": float(profile.LATITUDE.values.item()),
            "longitude": float(profile.LONGITUDE.values.item()),
            "direction": get_safe_string_from_xarray_var(profile.DIRECTION),
            "data_mode": get_safe_string_from_xarray_var(profile.DATA_MODE),
            "position_qc": get_safe_string_from_xarray_var(profile.POSITION_QC) if 'POSITION_QC' in profile else None,
            "positioning_system": get_safe_string_from_xarray_var(profile.POSITIONING_SYSTEM) if 'POSITIONING_SYSTEM' in profile else None,
            "positioning_system_2": get_safe_string_from_xarray_var(profile.POSITIONING_SYSTEM_2) if 'POSITIONING_SYSTEM_2' in profile else None,
            "position_accuracy_1": float(profile.POSITION_ACCURACY_1.values.item()) if 'POSITION_ACCURACY_1' in profile else None,
            "position_accuracy_2": float(profile.POSITION_ACCURACY_2.values.item()) if 'POSITION_ACCURACY_2' in profile else None,
            "vertical_sampling_scheme": get_safe_string_from_xarray_var(profile.VERTICAL_SAMPLING_SCHEME) if 'VERTICAL_SAMPLING_SCHEME' in profile else None,
            "config_mission_number": int(profile.CONFIG_MISSION_NUMBER.values.item()) if 'CONFIG_MISSION_NUMBER' in profile else None,
            "has_adjusted": any('_ADJUSTED' in var for var in profile.data_vars)
        }

        # Stats
        pres = profile.PRES.values if 'PRES' in profile else np.array([])
        temp = profile.TEMP.values if 'TEMP' in profile else np.array([])
        psal = profile.PSAL.values if 'PSAL' in profile else np.array([])

        valid_pres = pres[~np.isnan(pres)]
        valid_temp = temp[~np.isnan(temp)]
        valid_psal = psal[~np.isnan(psal)]

        metadata.update({
            "min_pres": float(valid_pres.min()) if len(valid_pres) > 0 else None,
            "max_pres": float(valid_pres.max()) if len(valid_pres) > 0 else None,
            "mean_pres": float(valid_pres.mean()) if len(valid_pres) > 0 else None,
            "min_temp": float(valid_temp.min()) if len(valid_temp) > 0 else None,
            "max_temp": float(valid_temp.max()) if len(valid_temp) > 0 else None,
            "mean_temp": float(valid_temp.mean()) if len(valid_temp) > 0 else None,
            "min_psal": float(valid_psal.min()) if len(valid_psal) > 0 else None,
            "max_psal": float(valid_psal.max()) if len(valid_psal) > 0 else None,
            "mean_psal": float(valid_psal.mean()) if len(valid_psal) > 0 else None,
            "profile_pres_qc": get_safe_string_from_xarray_var(profile.PRES_QC) if 'PRES_QC' in profile else None,
            "profile_temp_qc": get_safe_string_from_xarray_var(profile.TEMP_QC) if 'TEMP_QC' in profile else None,
            "profile_psal_qc": get_safe_string_from_xarray_var(profile.PSAL_QC) if 'PSAL_QC' in profile else None,
            "ocean_region": "Indian Ocean"  # Customize as needed
        })

        # Upsert to handle duplicates
        supabase.table("argo_metadata").upsert(metadata, on_conflict="float_id, cycle_number", ignore_duplicates=True).execute()
        print(f"Metadata inserted/updated: {float_id}-{cycle_number}")

# Call it
process_profile_metadata(ds, file_id)

Metadata inserted/updated: 6990715-31
Metadata inserted/updated: 5905384-297
Metadata inserted/updated: 6904144-139
Metadata inserted/updated: 5905521-132
Metadata inserted/updated: 6990575-5
Metadata inserted/updated: 1902598-76
Metadata inserted/updated: 5905771-266
Metadata inserted/updated: 5904815-336
Metadata inserted/updated: 5905678-294
Metadata inserted/updated: 3902337-46
Metadata inserted/updated: 7900961-149
Metadata inserted/updated: 6990643-1
Metadata inserted/updated: 1901839-337
Metadata inserted/updated: 4902950-279
Metadata inserted/updated: 5905779-258
Metadata inserted/updated: 3901829-305
Metadata inserted/updated: 1902266-200
Metadata inserted/updated: 6990588-65
Metadata inserted/updated: 6903088-116
Metadata inserted/updated: 5906395-204
Metadata inserted/updated: 5906621-185
Metadata inserted/updated: 5905475-198
Metadata inserted/updated: 5906395-203
Metadata inserted/updated: 5906622-185
Metadata inserted/updated: 7900878-40
Metadata inserted/updated: 1901749

In [41]:
def process_calibration(ds, file_id):
    if 'N_PARAM' in ds.sizes and ds.sizes['N_PARAM'] > 0:
        for i in range(ds.sizes['N_PROF']):
            profile = ds.isel(N_PROF=i)
            float_id = get_safe_string_from_xarray_var(profile.PLATFORM_NUMBER)
            cycle_number = int(profile.CYCLE_NUMBER.values.item())

            for p in range(ds.sizes['N_PARAM']):
                param_slice = ds.isel(N_PROF=i, N_PARAM=p)

                # PARAMETER
                param = get_safe_string_from_xarray_var(param_slice.PARAMETER)
                if not param or param == '_FillValue':
                    continue

                calib_data = {
                    "float_id": float_id,
                    "cycle_number": cycle_number,
                    "parameter": param,
                    "parameter_sensor": get_safe_string_from_xarray_var(param_slice.PARAMETER_SENSOR) if 'PARAMETER_SENSOR' in param_slice else None,
                    "calib_equation": get_safe_string_from_xarray_var(param_slice.SCIENTIFIC_CALIB_EQUATION) if 'SCIENTIFIC_CALIB_EQUATION' in param_slice else None,
                    "calib_coefficients": {},
                    "calib_comment": get_safe_string_from_xarray_var(param_slice.SCIENTIFIC_CALIB_COMMENT) if 'SCIENTIFIC_CALIB_COMMENT' in param_slice else None,
                    "calib_date": None,
                    "file_id": file_id
                }

                # Coefficients
                if 'SCIENTIFIC_CALIB_COEFFICIENTS' in param_slice:
                    coeffs_str = get_safe_string_from_xarray_var(param_slice.SCIENTIFIC_CALIB_COEFFICIENTS)
                    if coeffs_str and coeffs_str != '_FillValue':
                        try:
                            calib_data["calib_coefficients"] = json.loads(coeffs_str)
                        except:
                            calib_data["calib_coefficients"] = {"raw": coeffs_str}

                # Calib Date (more robust handling)
                if 'SCIENTIFIC_CALIB_DATE' in param_slice:
                    date_str = get_safe_string_from_xarray_var(param_slice.SCIENTIFIC_CALIB_DATE)
                    if date_str and date_str != '_FillValue':
                        try:
                            calib_data["calib_date"] = pd.to_datetime(date_str, format='%Y%m%d%H%M%S', errors='coerce').isoformat()
                        except ValueError:
                            calib_data["calib_date"] = None

                # The primary fix for the APIError is adding the unique constraint to the Supabase table.
                # This Python code assumes that unique constraint (float_id, cycle_number, parameter)
                # is correctly defined in the 'calibration_info' table in your Supabase project.
                supabase.table("calibration_info").upsert(calib_data, on_conflict="float_id, cycle_number, parameter", ignore_duplicates=True).execute()
                print(f"Calibration inserted/updated for {float_id}-{cycle_number}-{param}")

# Call it
process_calibration(ds, file_id)

Calibration inserted/updated for 6990715-31-PRES
Calibration inserted/updated for 6990715-31-TEMP
Calibration inserted/updated for 6990715-31-PSAL
Calibration inserted/updated for 5905384-297-PRES
Calibration inserted/updated for 5905384-297-TEMP
Calibration inserted/updated for 5905384-297-PSAL
Calibration inserted/updated for 6904144-139-PRES
Calibration inserted/updated for 6904144-139-TEMP
Calibration inserted/updated for 6904144-139-PSAL
Calibration inserted/updated for 5905521-132-PRES
Calibration inserted/updated for 5905521-132-TEMP
Calibration inserted/updated for 5905521-132-PSAL
Calibration inserted/updated for 6990575-5-PRES
Calibration inserted/updated for 6990575-5-TEMP
Calibration inserted/updated for 6990575-5-PSAL
Calibration inserted/updated for 1902598-76-PRES
Calibration inserted/updated for 1902598-76-TEMP
Calibration inserted/updated for 1902598-76-PSAL
Calibration inserted/updated for 5905771-266-PRES
Calibration inserted/updated for 5905771-266-TEMP
Calibration 

In [42]:
def process_history(ds, file_id):
    if 'N_HISTORY' in ds.sizes and ds.sizes['N_HISTORY'] > 0:
        for i in range(ds.sizes['N_PROF']):
            profile = ds.isel(N_PROF=i)
            float_id = str(profile.PLATFORM_NUMBER.values.item()).strip()
            cycle_number = int(profile.CYCLE_NUMBER.values.item())

            for h in range(ds.sizes['N_HISTORY']):
                hist_slice = ds.isel(N_PROF=i, N_HISTORY=h)

                # HISTORY_INSTITUTION (check if exists)
                if 'HISTORY_INSTITUTION' in hist_slice:
                    inst_raw = hist_slice.HISTORY_INSTITUTION.values
                    hist_institution = ''.join(inst_raw.astype(str)).strip()
                    if not hist_institution or hist_institution == '_FillValue':
                        continue
                else:
                    continue  # Skip if no history vars at all

                history_data = {
                    "float_id": float_id,
                    "cycle_number": cycle_number,
                    "history_institution": hist_institution,
                    "history_step": ''.join(hist_slice.HISTORY_STEP.values.astype(str)).strip() if 'HISTORY_STEP' in hist_slice else None,
                    "history_software": ''.join(hist_slice.HISTORY_SOFTWARE.values.astype(str)).strip() if 'HISTORY_SOFTWARE' in hist_slice else None,
                    "history_software_release": ''.join(hist_slice.HISTORY_SOFTWARE_RELEASE.values.astype(str)).strip() if 'HISTORY_SOFTWARE_RELEASE' in hist_slice else None,
                    "history_reference": ''.join(hist_slice.HISTORY_REFERENCE.values.astype(str)).strip() if 'HISTORY_REFERENCE' in hist_slice else None,
                    "history_action": ''.join(hist_slice.HISTORY_ACTION.values.astype(str)).strip() if 'HISTORY_ACTION' in hist_slice else None,
                    "history_parameter": ''.join(hist_slice.HISTORY_PARAMETER.values.astype(str)).strip() if 'HISTORY_PARAMETER' in hist_slice else None,
                    "history_start_pres": float(hist_slice.HISTORY_START_PRES.values.item()) if 'HISTORY_START_PRES' in hist_slice and not np.isnan(hist_slice.HISTORY_START_PRES.values.item()) else None,
                    "history_stop_pres": float(hist_slice.HISTORY_STOP_PRES.values.item()) if 'HISTORY_STOP_PRES' in hist_slice and not np.isnan(hist_slice.HISTORY_STOP_PRES.values.item()) else None,
                    "history_previous_value": ''.join(hist_slice.HISTORY_PREVIOUS_VALUE.values.astype(str)).strip() if 'HISTORY_PREVIOUS_VALUE' in hist_slice else None,
                    "history_qctest": ''.join(hist_slice.HISTORY_QCTEST.values.astype(str)).strip() if 'HISTORY_QCTEST' in hist_slice else None,
                    "history_qc1": str(hist_slice.HISTORY_QC1.values.item()).strip() if 'HISTORY_QC1' in hist_slice else None,
                    "history_qc2": str(hist_slice.HISTORY_QC2.values.item()).strip() if 'HISTORY_QC2' in hist_slice else None,
                    "file_id": file_id
                }

                # HISTORY_DATE
                if 'HISTORY_DATE' in hist_slice:
                    date_raw = hist_slice.HISTORY_DATE.values.item()
                    if not pd.isna(date_raw) and date_raw < 99999999999999:
                        try:
                            history_data["history_date"] = pd.to_datetime(str(int(date_raw)), format='%Y%m%d%H%M%S', errors='coerce').isoformat()
                        except:
                            pass

                supabase.table("history_info").insert(history_data).execute()
                print(f"History inserted for {float_id}-{cycle_number}-{h}")

# Call it
process_history(ds, file_id)

In [47]:
def get_safe_string_from_xarray_var(var):
    """Safely extract string from xarray variable, handling bytes, arrays, fill values"""
    if var is None:
        return None
    raw = var.values.item() if var.values.size == 1 else var.values
    if isinstance(raw, bytes):
        s = raw.decode('utf-8', errors='ignore')
    elif isinstance(raw, np.ndarray):
        s = ''.join(raw.astype(str))
    else:
        s = str(raw)
    s = s.strip()
    if s in ('', '_FillValue', 'nan'):
        return None
    return s

In [52]:
def safe_float_format(value, format_str):
    """Safely format a float with fallback to 'N/A' if None/NaN"""
    if value is None or pd.isna(value):
        return 'N/A'
    try:
        return format_str.format(value)
    except (ValueError, TypeError):
        return str(value)

def process_embeddings(ds):
    inserted = 0
    skipped = 0
    errors = 0

    for i in range(ds.sizes['N_PROF']):
        profile = ds.isel(N_PROF=i)

        # Safe float_id using your helper
        float_id = get_safe_string_from_xarray_var(profile.PLATFORM_NUMBER)
        if not float_id or float_id in ('_FillValue', ''):
            print(f"Profile {i}: Invalid/missing float_id, skipping")
            skipped += 1
            continue

        # Safe cycle_number
        cycle_raw = profile.CYCLE_NUMBER.values.item()
        try:
            cycle_number = int(cycle_raw)
        except (ValueError, TypeError):
            print(f"Profile {i}: Invalid cycle_number '{cycle_raw}', skipping")
            skipped += 1
            continue

        # Fetch metadata
        try:
            result = supabase.table("argo_metadata") \
                .select("*") \
                .eq("float_id", float_id) \
                .eq("cycle_number", cycle_number) \
                .execute()

            if not result.data:
                print(f"No metadata found for {float_id}-{cycle_number}. Skipping embedding.")
                skipped += 1
                continue

            metadata = result.data[0]
        except Exception as e:
            print(f"Metadata fetch failed for {float_id}-{cycle_number}: {e}")
            errors += 1
            continue

        # Build summary with safe formatting
        summary_parts = [
            f"Float: {metadata.get('platform_number', 'N/A')}, Cycle: {cycle_number}, Direction: {metadata.get('direction', 'N/A')}",
            f"Date: {metadata.get('juld', 'N/A')}, Lat/Lon: ({safe_float_format(metadata.get('latitude'), '{:.2f}')}, {safe_float_format(metadata.get('longitude'), '{:.2f}')})",
            f"Position QC: {metadata.get('position_qc', 'N/A')}, System: {metadata.get('positioning_system', 'N/A')}",
            f"Data Mode: {metadata.get('data_mode', 'N/A')}, Adjusted: {metadata.get('has_adjusted', 'N/A')}",
            f"Vertical Scheme: {metadata.get('vertical_sampling_scheme', 'N/A')}, Mission: {metadata.get('config_mission_number', 'N/A')}",
            f"Pres: {safe_float_format(metadata.get('mean_pres'), '{:.0f}')} dbar (range {safe_float_format(metadata.get('min_pres'), '{:.0f}')}-{safe_float_format(metadata.get('max_pres'), '{:.0f}')})",
            f"Temp: {safe_float_format(metadata.get('mean_temp'), '{:.2f}')}°C (range {safe_float_format(metadata.get('min_temp'), '{:.2f}')}-{safe_float_format(metadata.get('max_temp'), '{:.2f}')})",
            f"PSAL: {safe_float_format(metadata.get('mean_psal'), '{:.2f}')} PSU (range {safe_float_format(metadata.get('min_psal'), '{:.2f}')}-{safe_float_format(metadata.get('max_psal'), '{:.2f}')})",
            f"QC: Pres={metadata.get('profile_pres_qc', 'N/A')}, Temp={metadata.get('profile_temp_qc', 'N/A')}, PSAL={metadata.get('profile_psal_qc', 'N/A')}",
            f"Region: {metadata.get('ocean_region', 'Indian Ocean')}"
        ]

        # Filter out lines that are mostly useless
        full_summary = " | ".join([
            p for p in summary_parts
            if 'N/A' not in p or any(k in p.lower() for k in ['pres:', 'temp:', 'psal:'])  # keep stats even with N/A range
        ])

        if not full_summary.strip():
            print(f"Empty/invalid summary for {float_id}-{cycle_number}, skipping")
            skipped += 1
            continue

        # Generate embedding
        try:
            embedding = embedder.encode(full_summary).tolist()
        except Exception as e:
            print(f"Embedding generation failed for {float_id}-{cycle_number}: {e}")
            errors += 1
            continue

        embedding_data = {
            "float_id": float_id,
            "cycle_number": cycle_number,
            "summary": full_summary,
            "embedding": embedding
        }

        # Duplicate check + insert
        try:
            existing = supabase.table("argo_embeddings") \
                .select("float_id") \
                .limit(1) \
                .eq("float_id", float_id) \
                .eq("cycle_number", cycle_number) \
                .execute()

            if existing.data:
                print(f"Skipping duplicate embedding: {float_id}-{cycle_number}")
                skipped += 1
                continue

            supabase.table("argo_embeddings").insert(embedding_data).execute()
            print(f"Embedding inserted: {float_id}-{cycle_number}")
            inserted += 1
        except Exception as e:
            print(f"Insert failed for {float_id}-{cycle_number}: {e}")
            errors += 1

    print(f"\nEmbedding processing complete:")
    print(f"  Inserted: {inserted}")
    print(f"  Skipped (duplicates or missing data): {skipped}")
    print(f"  Errors: {errors}")

process_embeddings(ds)

Skipping duplicate embedding: 6990715-31
Skipping duplicate embedding: 5905384-297
Skipping duplicate embedding: 6904144-139
Skipping duplicate embedding: 5905521-132
Skipping duplicate embedding: 6990575-5
Skipping duplicate embedding: 1902598-76
Skipping duplicate embedding: 5905771-266
Skipping duplicate embedding: 5904815-336
Skipping duplicate embedding: 5905678-294
Skipping duplicate embedding: 3902337-46
Skipping duplicate embedding: 7900961-149
Skipping duplicate embedding: 6990643-1
Embedding inserted: 1901839-337
Embedding inserted: 4902950-279
Embedding inserted: 5905779-258
Embedding inserted: 3901829-305
Embedding inserted: 1902266-200
Embedding inserted: 6990588-65
Embedding inserted: 6903088-116
Embedding inserted: 5906395-204
Embedding inserted: 5906621-185
Embedding inserted: 5905475-198
Embedding inserted: 5906395-203
Embedding inserted: 5906622-185
Embedding inserted: 7900878-40
Embedding inserted: 1901749-219
Embedding inserted: 5905402-283
Embedding inserted: 29038

In [60]:
exists = supabase.table("argo_metadata") \
    .select("float_id") \
    .eq("float_id", float_id) \
    .eq("cycle_number", cycle_number) \
    .execute()

if not exists.data:
    print(f"Metadata missing for {float_id}-{cycle_number}, skipping index")
    skipped += 1



SyntaxError: 'continue' not properly in loop (ipython-input-2308401776.py, line 10)

In [62]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

def process_parquet_and_index(
    ds,
    bucket_name="argo-parquet",
    generate_signed_urls=False,
    signed_expiry_seconds=3600
):
    """
    Create Parquet files per profile and index them in profile_parquet_index.
    GUARANTEES DB ↔ storage consistency.
    """

    inserted = 0
    skipped = 0
    errors = 0
    signed_urls = [] if generate_signed_urls else None

    for i in range(ds.sizes["N_PROF"]):
        profile = ds.isel(N_PROF=i)

        # ---------- 1. Canonical IDs ----------
        float_id = get_safe_string_from_xarray_var(profile.PLATFORM_NUMBER)
        if not float_id:
            print(f"[SKIP] Profile {i}: missing float_id")
            skipped += 1
            continue

        float_id = float_id.strip()

        try:
            cycle_number = int(profile.CYCLE_NUMBER.values.item())
        except Exception:
            print(f"[SKIP] {float_id}: invalid cycle_number")
            skipped += 1
            continue

        # ---------- 2. Metadata MUST exist ----------
        meta_exists = supabase.table("argo_metadata") \
            .select("float_id") \
            .eq("float_id", float_id) \
            .eq("cycle_number", cycle_number) \
            .execute()

        if not meta_exists.data:
            print(f"[SKIP] Metadata missing for {float_id}-{cycle_number}")
            skipped += 1
            continue

        # ---------- 3. Storage path ----------
        basin = float_id[:2]
        cycle_str = f"{cycle_number:04d}"
        parquet_uri = f"by_float/{basin}/{float_id}/{cycle_str}.parquet"

        # ---------- 4. Variables ----------
        depth_vars = [
            "PRES", "PRES_QC",
            "PRES_ADJUSTED", "PRES_ADJUSTED_QC", "PRES_ADJUSTED_ERROR"
        ]

        core_vars = ["TEMP", "PSAL", "CNDC"]
        qc_vars = [v + "_QC" for v in core_vars]
        adj_vars = [v + "_ADJUSTED" for v in core_vars]
        adj_qc_vars = [v + "_ADJUSTED_QC" for v in core_vars]
        adj_err_vars = [v + "_ADJUSTED_ERROR" for v in core_vars]

        wanted_vars = depth_vars + core_vars + qc_vars + adj_vars + adj_qc_vars + adj_err_vars
        available_vars = [v for v in wanted_vars if v in profile]

        if not available_vars:
            print(f"[SKIP] No usable variables for {float_id}-{cycle_number}")
            skipped += 1
            continue

        # ---------- 5. Build DataFrame ----------
        df = pd.DataFrame({v: profile[v].values for v in available_vars})

        measurement_cols = [v for v in available_vars if "QC" not in v and "ERROR" not in v]
        if measurement_cols:
            df = df.dropna(how="all", subset=measurement_cols)

        if df.empty:
            print(f"[SKIP] Empty dataframe for {float_id}-{cycle_number}")
            skipped += 1
            continue

        # ---------- 6. Write Parquet ----------
        local_path = f"/tmp/{float_id}_{cycle_str}_{i}.parquet"
        pq.write_table(pa.Table.from_pandas(df), local_path)

        # ---------- 7. Upload ----------
        try:
            with open(local_path, "rb") as f:
                supabase.storage.from_(bucket_name).upload(parquet_uri, f)
        except Exception as e:
            print(f"[ERROR] Upload failed {float_id}-{cycle_number}: {e}")
            errors += 1
            continue

        # ---------- 8. Optional signed URL ----------
        if generate_signed_urls:
            try:
                url = supabase.storage.from_(bucket_name).create_signed_url(
                    parquet_uri, expires_in=signed_expiry_seconds
                )
                signed_urls.append({
                    "float_id": float_id,
                    "cycle_number": cycle_number,
                    "signed_url": url
                })
            except Exception:
                pass

        # ---------- 9. Index insert ----------
        pres = df["PRES"] if "PRES" in df.columns else pd.Series(dtype=float)

        index_row = {
            "float_id": float_id,
            "cycle_number": cycle_number,
            "parquet_uri": parquet_uri,
            "row_count": int(len(df)),
            "min_pres": float(pres.min()) if not pres.empty else None,
            "max_pres": float(pres.max()) if not pres.empty else None,
            "max_depth": float(pres.max()) if not pres.empty else None,
            "variables": available_vars,
            "file_size_bytes": os.path.getsize(local_path)
        }

        try:
            supabase.table("profile_parquet_index").upsert(
                index_row,
                on_conflict="float_id, cycle_number"
            ).execute()

            print(f"[OK] Indexed {float_id}-{cycle_number}")
            inserted += 1

        except Exception as e:
            print(f"[ERROR] Index failed {float_id}-{cycle_number}: {e}")
            errors += 1

    # ---------- Final report ----------
    print("\n=== Parquet & Index Summary ===")
    print(f"Inserted : {inserted}")
    print(f"Skipped  : {skipped}")
    print(f"Errors   : {errors}")

    if generate_signed_urls:
        return {
            "inserted": inserted,
            "skipped": skipped,
            "errors": errors,
            "signed_urls": signed_urls
        }

    return {
        "inserted": inserted,
        "skipped": skipped,
        "errors": errors
    }

stats = process_parquet_and_index(ds)
print(stats)
print("SUCESS!!")


[OK] Indexed 6990715-31
[OK] Indexed 5905384-297
[OK] Indexed 6904144-139
[OK] Indexed 5905521-132
[OK] Indexed 6990575-5
[OK] Indexed 1902598-76
[OK] Indexed 5905771-266
[OK] Indexed 5904815-336
[OK] Indexed 5905678-294
[OK] Indexed 3902337-46
[OK] Indexed 7900961-149
[OK] Indexed 6990643-1
[OK] Indexed 1901839-337
[OK] Indexed 4902950-279
[OK] Indexed 5905779-258
[OK] Indexed 3901829-305
[OK] Indexed 1902266-200
[OK] Indexed 6990588-65
[OK] Indexed 6903088-116
[OK] Indexed 5906395-204
[OK] Indexed 5906621-185
[OK] Indexed 5905475-198
[OK] Indexed 5906395-203
[OK] Indexed 5906622-185
[OK] Indexed 7900878-40
[OK] Indexed 1901749-219
[OK] Indexed 5905402-283
[OK] Indexed 2903826-2
[OK] Indexed 7902148-24
[OK] Indexed 1902523-69
[OK] Indexed 5907171-32
[OK] Indexed 7900939-112
[OK] Indexed 2903892-84
[OK] Indexed 7902243-41
[OK] Indexed 4903869-31
[OK] Indexed 5907139-41
[OK] Indexed 6990608-98
[OK] Indexed 5905269-303
[OK] Indexed 4903777-71
[OK] Indexed 3902658-28
[OK] Indexed 1902505-

In [5]:
# Install missing package if not already installed (for standalone execution)
!pip install supabase

import xarray as xr
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from supabase import create_client, Client
from sentence_transformers import SentenceTransformer
from datetime import datetime, timedelta
import json
from google.colab import userdata
import numpy as np

# Initialize Supabase (use your secrets)
supabase_url = userdata.get('SUPABASE_URL')
supabase_key = userdata.get('SUPABASE_ANON_KEY')
supabase: Client = create_client(supabase_url, supabase_key)

print("Initialization complete!")
# Initialize embedder here to ensure it's defined
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def embed_user_query(query: str) -> list:
    """
    Convert user query text into embedding vector.
    """
    embedding = embedder.encode(query)
    return embedding.tolist()

def fetch_context_from_vdb(
    query_embedding: list,
    top_k: int = 5
):
    """
    Fetch top-K most similar summaries from argo_embeddings.
    """

    response = supabase.rpc(
        "match_argo_embeddings",
        {
            "query_embedding": query_embedding,
            "match_count": top_k
        }
    ).execute()

    return response.data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 9.3 MB/s eta 0:00:00
Initialization complete!


In [6]:
user_query = "Show temperature variation below 500m for Indian Ocean floats"

# 1. Embed query
query_embedding = embed_user_query(user_query)

# 2. Fetch vector context
contexts = fetch_context_from_vdb(query_embedding, top_k=5)

# 3. Print results
for c in contexts:
    print(
        f"Float {c['float_id']} Cycle {c['cycle_number']} "
        f"(similarity={c['similarity']:.3f})"
    )
    print(c["summary"])
    print("------")

Float 6990643 Cycle 1 (similarity=0.588)
Float: 6990643, Cycle: 1, Direction: A | Date: 2025-12-31T19:37:55+00:00, Lat/Lon: (-18.01, 105.03) | Position QC: 1, System: GPS | Data Mode: R, Adjusted: True | Vertical Scheme: Primary sampling: averaged [nominal   2 dbar binned data sampled at 1.0 Hz from a SBE41CP; bin detail from 0 dbar (number bins/bin width):   10/  1; 500/  2;remaining/  2], Mission: 1 | Pres: 492 dbar (range 1-994) | Temp: 12.09°C (range 5.21-26.97) | PSAL: 34.77 PSU (range 34.36-35.65) | QC: Pres=1, Temp=1, PSAL=1 | Region: Indian Ocean
------
Float 1901761 Cycle 175 (similarity=0.586)
Float: 1901761, Cycle: 175, Direction: A | Date: 2025-12-31T07:59:31+00:00, Lat/Lon: (-34.88, 36.55) | Position QC: 1, System: GPS | Data Mode: A, Adjusted: True | Vertical Scheme: Primary sampling: averaged [], Mission: 3 | Pres: 980 dbar (range 4-1955) | Temp: 9.25°C (range 2.96-20.89) | PSAL: 34.89 PSU (range 34.41-35.66) | QC: Pres=1, Temp=1, PSAL=1 | Region: Indian Ocean
------
Flo